# Real Data ST Mean-Design Check: Hybrid vs Column V3

This notebook tests whether richer GLS mean/detrending reduces the Hybrid-vs-Column difference in the spatio-temporal model.

Models compared for one real day:

- `HybridSTTrend_L08F04_C4F03_Op0p063_exactloc`: existing hybrid temporal conditioning, covariance uses `Source_Latitude` / `Source_Longitude`.
- `ColumnSTTrend_Up2_Right3_Down14_Lag2_head0_realloc`: reverse-L column temporal conditioning, scan/order uses regular grid, covariance uses `Source_Latitude` / `Source_Longitude`.

Mean designs compared:

- `base`: intercept + centered latitude + hour dummies
- `latlon`: intercept + centered latitude + centered longitude + hour dummies
- `hour_spatial`: hour-specific intercept + hour-specific latitude slope + hour-specific longitude slope

This is the direct ST counterpart of the pure-space mean-design test.


In [1]:
import sys, time, gc
from pathlib import Path

import numpy as np
import pandas as pd
import torch

LOCAL_SRC = "/Users/joonwonlee/Documents/GEMS_TCO-1/src"
AMAREL_SRC = "/home/jl2815/tco"
SRC = AMAREL_SRC if Path(AMAREL_SRC).exists() else LOCAL_SRC
sys.path.insert(0, SRC)

from GEMS_TCO import configuration as config
from GEMS_TCO.data_loader import load_data_dynamic_processed
from GEMS_TCO.kernels_st_trend_050726 import HybridSTTrendVecchiaFit, ColumnSTTrendVecchiaFit

DEVICE = torch.device('cpu')
DTYPE = torch.float64
ROUND_DECIMALS = 4

print('SRC:', SRC)
print('torch:', torch.__version__)
print('device:', DEVICE)


SRC: /Users/joonwonlee/Documents/GEMS_TCO-1/src
torch: 2.5.1
device: cpu


In [2]:
# Experiment config
YEAR = '2024'
MONTH = 7
DAY_IDX_LIST = [2]  # 0-based: [2] = July 3
LAT_RANGE = [-3, 2]
LON_RANGE = [121, 131]
MM_COND_NUMBER = 100
FIT_SMOOTHS = [0.5]
MEAN_DESIGNS = ['base', 'latlon', 'hour_spatial']

HYBRID_SPEC = {
    'model': 'HybridSTTrend_L08F04_C4F03_Op0p063_exactloc',
    'nheads': 0,
    'limit_A': 20,
    'lag1_local_count': 8,
    'lag1_fresh_count': 4,
    'lag2_local_count': 4,
    'lag2_fresh_count': 3,
    'daily_stride': 2,
    'lag1_lon_offset': 0.063,
    'spatial_coords_for_shift': None,
}

COLUMN_SPEC = {
    'model': 'ColumnSTTrend_Up2_Right3_Down14_Lag2_head0_realloc',
    'head_right_cols': 0,
    'above_count': 2,
    'right_col_count': 3,
    'per_lag_conditioning_count': 14,
    'lag_count': 2,
    'include_lag_self': False,
    'target_chunk_size': 512 if DEVICE.type == 'cpu' else 4096,
}

LBFGS_LR = 1.0
LBFGS_STEPS = 5
LBFGS_EVAL = 15
LBFGS_HIST = 10

INIT_DICT = {
    'sigmasq':    13.059,
    'range_lat':  0.2,
    'range_lon':  0.25,
    'range_time': 1.5,
    'advec_lat':  0.0218,
    'advec_lon': -0.1689,
    'nugget':     0.247,
}
P_LABELS = ['sigmasq', 'range_lat', 'range_lon', 'range_time', 'advec_lat', 'advec_lon', 'nugget']

OUT_DIR = Path('/Users/joonwonlee/Documents/GEMS_TCO-1/Exercises/st_model/day/local_computer/testing/log')
OUT_DIR.mkdir(parents=True, exist_ok=True)
OUT_CSV = OUT_DIR / 'real_st_mean_design_hybrid_colv3_realloc_cpu_050726_results.csv'

print('days:', DAY_IDX_LIST)
print('mean designs:', MEAN_DESIGNS)
print('output:', OUT_CSV)
print('hybrid:', HYBRID_SPEC)
print('column:', COLUMN_SPEC)


days: [2]
mean designs: ['base', 'latlon', 'hour_spatial']
output: /Users/joonwonlee/Documents/GEMS_TCO-1/Exercises/st_model/day/local_computer/testing/log/real_st_mean_design_hybrid_colv3_realloc_cpu_050726_results.csv
hybrid: {'model': 'HybridSTTrend_L08F04_C4F03_Op0p063_exactloc', 'nheads': 0, 'limit_A': 20, 'lag1_local_count': 8, 'lag1_fresh_count': 4, 'lag2_local_count': 4, 'lag2_fresh_count': 3, 'daily_stride': 2, 'lag1_lon_offset': 0.063, 'spatial_coords_for_shift': None}
column: {'model': 'ColumnSTTrend_Up2_Right3_Down14_Lag2_head0_realloc', 'head_right_cols': 0, 'above_count': 2, 'right_col_count': 3, 'per_lag_conditioning_count': 14, 'lag_count': 2, 'include_lag_self': False, 'target_chunk_size': 512}


In [3]:
def phys_to_log(d):
    phi2 = 1.0 / d['range_lon']
    phi1 = d['sigmasq'] * phi2
    phi3 = (d['range_lon'] / d['range_lat']) ** 2
    phi4 = (d['range_lon'] / d['range_time']) ** 2
    return [np.log(phi1), np.log(phi2), np.log(phi3), np.log(phi4), d['advec_lat'], d['advec_lon'], np.log(d['nugget'])]


def backmap_params(out_params):
    p = [float(x.item() if isinstance(x, torch.Tensor) else x) for x in out_params[:7]]
    phi2 = np.exp(p[1])
    phi3 = np.exp(p[2])
    phi4 = np.exp(p[3])
    rlon = 1.0 / phi2
    return {
        'sigmasq': np.exp(p[0]) / phi2,
        'range_lat': rlon / phi3 ** 0.5,
        'range_lon': rlon,
        'range_time': rlon / phi4 ** 0.5,
        'advec_lat': p[4],
        'advec_lon': p[5],
        'nugget': np.exp(p[6]),
    }


def make_params():
    return [torch.tensor([v], device=DEVICE, dtype=DTYPE, requires_grad=True) for v in phys_to_log(INIT_DICT)]


def count_valid(day_map):
    return sum(int((~torch.isnan(v[:, 2])).sum().item()) for v in day_map.values())


def empty_device_cache():
    gc.collect()
    if DEVICE.type == 'cuda':
        torch.cuda.empty_cache()


def hybrid_tail_count(model):
    total = 0
    for x in (getattr(model, 'X_A', None), getattr(model, 'X_AB', None), getattr(model, 'X_ABC', None)):
        if x is not None:
            total += int(x.shape[0])
    return total


def st_diag(model):
    batched = getattr(model, 'Batched_Groups', None)
    if batched:
        group_sizes = np.asarray([int(g['target_idx'].shape[0]) for g in batched], dtype=np.int64)
        m_sizes = np.asarray([int(g['max_m']) for g in batched], dtype=np.int64)
        return {
            'n_batches': int(len(batched)),
            'largest_batch_n': int(group_sizes.max()),
            'mean_m': float(m_sizes.mean()),
            'max_m': int(m_sizes.max()),
            'n_tails': int(group_sizes.sum()),
            'n_templates': np.nan,
        }
    return {
        'n_batches': np.nan,
        'largest_batch_n': np.nan,
        'mean_m': np.nan,
        'max_m': np.nan,
        'n_tails': hybrid_tail_count(model),
        'n_templates': np.nan,
    }


def round_df(df, digits=ROUND_DECIMALS):
    out = df.copy()
    num_cols = out.select_dtypes(include=[np.number]).columns
    out[num_cols] = out[num_cols].round(digits)
    return out


def save_results(rows):
    df = pd.DataFrame(rows)
    round_df(df).to_csv(OUT_CSV, index=False, float_format=f'%.{ROUND_DECIMALS}f')
    return df


def result_row(date_str, day_idx, smooth, mean_design, model_name, kernel, loss, fit_iter, pre_s, fit_s, n_valid, est, diag):
    row = {
        'date_str': date_str,
        'day_idx': int(day_idx),
        'smooth': float(smooth),
        'mean_design': mean_design,
        'model': model_name,
        'kernel': kernel,
        'loss': float(loss),
        'fit_iter_raw': int(fit_iter),
        'fit_steps_reported': int(fit_iter) + 1,
        'precompute_s': round(pre_s, 3),
        'fit_s': round(fit_s, 3),
        'total_s': round(pre_s + fit_s, 3),
        'n_valid': int(n_valid),
    }
    row.update({f'est_{k}': est[k] for k in P_LABELS})
    row.update(diag)
    return row

print('Initial log params:', [round(x, 4) for x in phys_to_log(INIT_DICT)])


Initial log params: [3.9558, 1.3863, 0.4463, -3.5835, 0.0218, -0.1689, -1.3984]


In [4]:
# Load full month once.
loader = load_data_dynamic_processed(config.mac_data_load_path)

df_map, ord_mm, nns_map_full, monthly_mean = loader.load_maxmin_ordered_data_bymonthyear(
    lat_lon_resolution=[1, 1],
    mm_cond_number=MM_COND_NUMBER,
    years_=[YEAR],
    months_=[MONTH],
    lat_range=LAT_RANGE,
    lon_range=LON_RANGE,
    is_whittle=False,
)

sorted_month_keys = sorted(df_map.keys())
day_key_map = {
    day_idx: sorted_month_keys[day_idx * 8:(day_idx + 1) * 8]
    for day_idx in DAY_IDX_LIST
}
selected_keys = [k for day_idx in DAY_IDX_LIST for k in day_key_map[day_idx]]
df_map_selected = {k: df_map[k] for k in selected_keys}

first_key = selected_keys[0]
first_df = df_map_selected[first_key].iloc[ord_mm].reset_index(drop=True)
grid_coords_ordered = first_df[['Latitude', 'Longitude']].values.astype(np.float64)

del df_map
gc.collect()

print(f'Monthly mean TCO: {monthly_mean:.4f}')
print(f'Month time slots loaded then trimmed: {len(sorted_month_keys)} -> {len(selected_keys)}')
print(f'Grid cells: {len(grid_coords_ordered):,}')
print('grid lat:', grid_coords_ordered[:, 0].min(), grid_coords_ordered[:, 0].max())
print('grid lon:', grid_coords_ordered[:, 1].min(), grid_coords_ordered[:, 1].max())


--- Global Monthly Mean for 2024-7: 257.9726 ---
--- Generating NNS Map for Vecchia (C++ Accelerated) ---
Monthly mean TCO: 257.9726
Month time slots loaded then trimmed: 248 -> 8
Grid cells: 18,126
grid lat: -2.9720000000000044 2.0
grid lon: 121.04600000000188 131.0


In [5]:
def load_day_map(day_idx, keep_ori=True):
    day_keys = day_key_map[day_idx]
    day_df_map = {k: df_map_selected[k] for k in day_keys}
    hour_indices = [0, len(day_keys)]
    day_map, _ = loader.load_working_data(
        day_df_map, monthly_mean, hour_indices,
        ord_mm=ord_mm, dtype=DTYPE, keep_ori=keep_ori,
    )
    return {k: v.to(DEVICE) for k, v in day_map.items()}, day_keys


def fit_hybrid_st(day_idx, smooth, mean_design):
    date_str = f'{YEAR}{MONTH:02d}{day_idx + 1:02d}'
    day_map, day_keys = load_day_map(day_idx, keep_ori=True)
    n_valid = count_valid(day_map)
    print('\n' + '=' * 80)
    print(f"HYBRID ST | mean={mean_design} | DAY {date_str} | smooth={smooth} | valid={n_valid:,}")

    params = make_params()
    model = HybridSTTrendVecchiaFit(
        smooth=smooth,
        input_map=day_map,
        nns_map=nns_map_full,
        mm_cond_number=MM_COND_NUMBER,
        nheads=HYBRID_SPEC['nheads'],
        limit_A=HYBRID_SPEC['limit_A'],
        limit_B_local=HYBRID_SPEC['lag1_local_count'],
        limit_C_local=HYBRID_SPEC['lag2_local_count'],
        daily_stride=HYBRID_SPEC['daily_stride'],
        spatial_coords=HYBRID_SPEC['spatial_coords_for_shift'],
        lag1_lon_offset=HYBRID_SPEC['lag1_lon_offset'],
        lag1_fresh_count=HYBRID_SPEC['lag1_fresh_count'],
        lag2_fresh_count=HYBRID_SPEC['lag2_fresh_count'],
        mean_design=mean_design,
    )

    t_pre = time.time()
    model.precompute_conditioning_sets()
    pre_s = time.time() - t_pre
    diag = st_diag(model)

    opt = model.set_optimizer(params, lr=LBFGS_LR, max_iter=LBFGS_EVAL, max_eval=LBFGS_EVAL, history_size=LBFGS_HIST)
    t_fit = time.time()
    out, fit_iter = model.fit_vecc_lbfgs(params, opt, max_steps=LBFGS_STEPS, grad_tol=1e-5)
    fit_s = time.time() - t_fit
    est = backmap_params(out)

    row = result_row(date_str, day_idx, smooth, mean_design, HYBRID_SPEC['model'], 'hybrid_st_trend_exactloc', float(out[-1]), fit_iter, pre_s, fit_s, n_valid, est, diag)
    row.update({
        'total_conditioning_nominal': 41,
        'coords_used_for_covariance': 'Source_Latitude/Source_Longitude',
        'coords_used_for_shift_lookup': 'exact_t0_source_coords',
    })
    print('RESULT:', row)
    del model, day_map, params, opt
    empty_device_cache()
    return row


def fit_column_st(day_idx, smooth, mean_design):
    date_str = f'{YEAR}{MONTH:02d}{day_idx + 1:02d}'
    # keep_ori=True: covariance uses real source coordinates. grid_coords only defines reverse-L scan/order.
    day_map, day_keys = load_day_map(day_idx, keep_ori=True)
    n_valid = count_valid(day_map)
    print('\n' + '=' * 80)
    print(f"COLUMN ST REALLOC | mean={mean_design} | DAY {date_str} | smooth={smooth} | valid={n_valid:,}")

    params = make_params()
    model = ColumnSTTrendVecchiaFit(
        smooth=smooth,
        input_map=day_map,
        mm_cond_number=MM_COND_NUMBER,
        grid_coords=grid_coords_ordered,
        head_right_cols=COLUMN_SPEC['head_right_cols'],
        above_count=COLUMN_SPEC['above_count'],
        right_col_count=COLUMN_SPEC['right_col_count'],
        per_lag_conditioning_count=COLUMN_SPEC['per_lag_conditioning_count'],
        lag_count=COLUMN_SPEC['lag_count'],
        include_lag_self=COLUMN_SPEC['include_lag_self'],
        target_chunk_size=COLUMN_SPEC['target_chunk_size'],
        use_data_coords_for_offsets=True,
        mean_design=mean_design,
    )

    t_pre = time.time()
    model.precompute_conditioning_sets()
    pre_s = time.time() - t_pre
    diag = st_diag(model)

    opt = model.set_optimizer(params, lr=LBFGS_LR, max_iter=LBFGS_EVAL, max_eval=LBFGS_EVAL, history_size=LBFGS_HIST)
    t_fit = time.time()
    out, fit_iter = model.fit_vecc_lbfgs(params, opt, max_steps=LBFGS_STEPS, grad_tol=1e-5)
    fit_s = time.time() - t_fit
    est = backmap_params(out)

    row = result_row(date_str, day_idx, smooth, mean_design, COLUMN_SPEC['model'], 'column_st_trend_realloc', float(out[-1]), fit_iter, pre_s, fit_s, n_valid, est, diag)
    row.update({
        'total_conditioning_nominal': COLUMN_SPEC['above_count'] + COLUMN_SPEC['per_lag_conditioning_count'] * (COLUMN_SPEC['lag_count'] + 1),
        'coords_used_for_covariance': 'Source_Latitude/Source_Longitude',
        'coords_used_for_shift_lookup': 'regular_grid_reverse_l_scan',
        'head_right_cols': COLUMN_SPEC['head_right_cols'],
        'above_count': COLUMN_SPEC['above_count'],
        'right_col_count': COLUMN_SPEC['right_col_count'],
        'per_lag_conditioning_count': COLUMN_SPEC['per_lag_conditioning_count'],
        'lag_count': COLUMN_SPEC['lag_count'],
    })
    print('RESULT:', row)
    del model, day_map, params, opt
    empty_device_cache()
    return row


In [6]:
# Run ST mean-design comparison.
rows = []
for day_idx in DAY_IDX_LIST:
    for smooth in FIT_SMOOTHS:
        for mean_design in MEAN_DESIGNS:
            rows.append(fit_hybrid_st(day_idx, smooth, mean_design))
            save_results(rows)
            rows.append(fit_column_st(day_idx, smooth, mean_design))
            save_results(rows)

results = pd.DataFrame(rows)
print('Saved:', OUT_CSV)
display(round_df(results))



HYBRID ST | mean=base | DAY 20240703 | smooth=0.5 | valid=140,352
Pre-computing HybridVecchia (smooth=0.5) [A=20, AB=33, ABC=41, B=local8+fresh4, C=local4+fresh3, lag1_offset=0.0630, stored=1]... [Mean Lat: -0.5041] [SetC=True] Done. (Heads=0, Tails A/AB/ABC=18067/17670/104615)
--- Starting Batched L-BFGS Optimization (GPU) ---
--- Step 1/5 / Loss: 1.555278 ---
  Param 0: Value=4.2505, Grad=0.010382663637184004
  Param 1: Value=1.5613, Grad=0.0014376351918495268
  Param 2: Value=0.0384, Grad=0.003425286673035978
  Param 3: Value=-4.6151, Grad=-0.010026535553011515
  Param 4: Value=-0.0786, Grad=-0.01971423988216795
  Param 5: Value=-0.2303, Grad=0.0633132555984851
  Param 6: Value=0.4722, Grad=-0.0008601189999463198
  Max Abs Grad: 6.331326e-02
------------------------------
--- Step 2/5 / Loss: 1.501821 ---
  Param 0: Value=3.6528, Grad=0.00036978783261233314
  Param 1: Value=0.9803, Grad=-0.0005317899593360942
  Param 2: Value=0.3782, Grad=8.201508921654366e-05
  Param 3: Value=-3.5

,date_str,day_idx,smooth,mean_design,model,kernel,loss,fit_iter_raw,fit_steps_reported,precompute_s,...,n_tails,n_templates,total_conditioning_nominal,coords_used_for_covariance,coords_used_for_shift_lookup,head_right_cols,above_count,right_col_count,per_lag_conditioning_count,lag_count
0,20240703,2,0.5,base,HybridSTTrend_L08F04_C4F03_Op0p063_exactloc,hybrid_st_trend_exactloc,1.4929,4,5,8.454,...,140352,NaN,41,Source_Latitude/Source_Longitude,exact_t0_source_coords,NaN,NaN,NaN,NaN,NaN
1,20240703,2,0.5,base,ColumnSTTrend_Up2_Right3_Down14_Lag2_head0_rea...,column_st_trend_realloc,1.3823,3,4,2.002,...,140352,NaN,44,Source_Latitude/Source_Longitude,regular_grid_reverse_l_scan,0.0,2.0,3.0,14.0,2.0
2,20240703,2,0.5,latlon,HybridSTTrend_L08F04_C4F03_Op0p063_exactloc,hybrid_st_trend_exactloc,1.4916,2,3,8.843,...,140352,NaN,41,Source_Latitude/Source_Longitude,exact_t0_source_coords,NaN,NaN,NaN,NaN,NaN
3,20240703,2,0.5,latlon,ColumnSTTrend_Up2_Right3_Down14_Lag2_head0_rea...,column_st_trend_realloc,1.3820,3,4,1.994,...,140352,NaN,44,Source_Latitude/Source_Longitude,regular_grid_reverse_l_scan,0.0,2.0,3.0,14.0,2.0
4,20240703,2,0.5,hour_spatial,HybridSTTrend_L08F04_C4F03_Op0p063_exactloc,hybrid_st_trend_exactloc,1.4912,2,3,8.812,...,140352,NaN,41,Source_Latitude/Source_Longitude,exact_t0_source_coords,NaN,NaN,NaN,NaN,NaN
5,20240703,2,0.5,hour_spatial,ColumnSTTrend_Up2_Right3_Down14_Lag2_head0_rea...,column_st_trend_realloc,1.3818,3,4,2.070,...,140352,NaN,44,Source_Latitude/Source_Longitude,regular_grid_reverse_l_scan,0.0,2.0,3.0,14.0,2.0


In [ ]:
summary_cols = [
    'date_str', 'smooth', 'mean_design', 'model', 'kernel', 'total_conditioning_nominal',
    'loss', 'precompute_s', 'fit_s', 'total_s', 'n_valid', 'n_batches', 'n_tails',
    'mean_m', 'max_m', 'largest_batch_n',
    'est_sigmasq', 'est_range_lat', 'est_range_lon', 'est_range_time', 'est_advec_lat', 'est_advec_lon', 'est_nugget',
    'coords_used_for_covariance', 'coords_used_for_shift_lookup',
]
existing = [c for c in summary_cols if c in results.columns]
display(round_df(results[existing].sort_values(['mean_design', 'model'])))

param_rows = []
for _, row in results.iterrows():
    for p in P_LABELS:
        param_rows.append({
            'date_str': row['date_str'],
            'mean_design': row['mean_design'],
            'model': row['model'],
            'parameter': p, 
            'estimate': row[f'est_{p}'],
        })
param_est = pd.DataFrame(param_rows)
display(round_df(param_est))


,date_str,smooth,mean_design,model,kernel,total_conditioning_nominal,loss,precompute_s,fit_s,total_s,...,largest_batch_n,est_sigmasq,est_range_lat,est_range_lon,est_range_time,est_advec_lat,est_advec_lon,est_nugget,coords_used_for_covariance,coords_used_for_shift_lookup
1,20240703,0.5,base,ColumnSTTrend_Up2_Right3_Down14_Lag2_head0_rea...,column_st_trend_realloc,44,1.3823,2.002,246.939,248.941,...,104615.0,12.8296,0.1510,0.1783,1.3561,-0.0710,-0.2356,1.2869,Source_Latitude/Source_Longitude,regular_grid_reverse_l_scan
0,20240703,0.5,base,HybridSTTrend_L08F04_C4F03_Op0p063_exactloc,hybrid_st_trend_exactloc,41,1.4929,8.454,261.198,269.652,...,NaN,14.2171,0.3078,0.3710,2.1862,-0.0644,-0.2598,2.7610,Source_Latitude/Source_Longitude,exact_t0_source_coords
5,20240703,0.5,hour_spatial,ColumnSTTrend_Up2_Right3_Down14_Lag2_head0_rea...,column_st_trend_realloc,44,1.3818,2.070,259.403,261.473,...,104615.0,12.4163,0.1437,0.1698,1.2904,-0.0683,-0.2343,1.2488,Source_Latitude/Source_Longitude,regular_grid_reverse_l_scan
4,20240703,0.5,hour_spatial,HybridSTTrend_L08F04_C4F03_Op0p063_exactloc,hybrid_st_trend_exactloc,41,1.4912,8.812,260.612,269.424,...,NaN,12.4887,0.2503,0.3032,1.8253,-0.0632,-0.2568,2.6088,Source_Latitude/Source_Longitude,exact_t0_source_coords
3,20240703,0.5,latlon,ColumnSTTrend_Up2_Right3_Down14_Lag2_head0_rea...,column_st_trend_realloc,44,1.3820,1.994,269.429,271.423,...,104615.0,12.5246,0.1456,0.1721,1.2994,-0.0692,-0.2353,1.2593,Source_Latitude/Source_Longitude,regular_grid_reverse_l_scan
2,20240703,0.5,latlon,HybridSTTrend_L08F04_C4F03_Op0p063_exactloc,hybrid_st_trend_exactloc,41,1.4916,8.843,248.683,257.526,...,NaN,12.6482,0.2556,0.3096,1.8367,-0.0635,-0.2563,2.6239,Source_Latitude/Source_Longitude,exact_t0_source_coords


,date_str,mean_design,model,parameter,estimate
0,20240703,base,HybridSTTrend_L08F04_C4F03_Op0p063_exactloc,sigmasq,14.2171
1,20240703,base,HybridSTTrend_L08F04_C4F03_Op0p063_exactloc,range_lat,0.3078
2,20240703,base,HybridSTTrend_L08F04_C4F03_Op0p063_exactloc,range_lon,0.3710
3,20240703,base,HybridSTTrend_L08F04_C4F03_Op0p063_exactloc,range_time,2.1862
4,20240703,base,HybridSTTrend_L08F04_C4F03_Op0p063_exactloc,advec_lat,-0.0644
5,20240703,base,HybridSTTrend_L08F04_C4F03_Op0p063_exactloc,advec_lon,-0.2598
6,20240703,base,HybridSTTrend_L08F04_C4F03_Op0p063_exactloc,nugget,2.7610
7,20240703,base,ColumnSTTrend_Up2_Right3_Down14_Lag2_head0_rea...,sigmasq,12.8296
8,20240703,base,ColumnSTTrend_Up2_Right3_Down14_Lag2_head0_rea...,range_lat,0.1510
9,20240703,base,ColumnSTTrend_Up2_Right3_Down14_Lag2_head0_rea...,range_lon,0.1783
